# 02 – SQL Analysis
## Reel Insights: Global Movie Industry (TMDB)

Load cleaned CSV files into SQLite, create relational tables, validate constraints and answer the 10 SQL analysis questions.


## 1. Imports and paths

In [ ]:
import os, sqlite3, pandas as pd
PROJECT_DIR = os.path.abspath("..") if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
DATA_DIR = os.path.join(PROJECT_DIR, "data")
CLEANED_DIR = os.path.join(DATA_DIR, "cleaned")
DB_PATH = os.path.join(DATA_DIR, "tmdb.db")

## 2. Load cleaned CSV files

In [ ]:
tables = ["movies","genres","movie_genres","cast","crew","movie_keywords"]
data = {}
for table in tables:
    path = os.path.join(CLEANED_DIR, table + "_cleaned.csv")
    if not os.path.exists(path):
        raise FileNotFoundError(f"{path} not found. Run Notebook 01 first.")
    data[table] = pd.read_csv(path)
    print(table, data[table].shape)

## 3. Create SQLite database and tables

In [ ]:
# Rebuild database for a reproducible run.
if os.path.exists(DB_PATH):
    os.remove(DB_PATH)
conn = sqlite3.connect(DB_PATH)
conn.execute("PRAGMA foreign_keys=ON")

ddl = [
'''CREATE TABLE movies (
movie_id INTEGER PRIMARY KEY, title TEXT, original_language TEXT,
release_date TEXT, runtime REAL, budget REAL, revenue REAL, popularity REAL,
vote_average REAL, vote_count INTEGER, status TEXT, overview TEXT,
financial_data_available INTEGER)''',
'''CREATE TABLE genres (genre_id INTEGER PRIMARY KEY, genre_name TEXT)''',
'''CREATE TABLE movie_genres (
movie_id INTEGER, genre_id INTEGER, PRIMARY KEY(movie_id,genre_id),
FOREIGN KEY(movie_id) REFERENCES movies(movie_id),
FOREIGN KEY(genre_id) REFERENCES genres(genre_id))''',
'''CREATE TABLE cast (
cast_row_id INTEGER PRIMARY KEY, movie_id INTEGER, person_id INTEGER,
actor_name TEXT, character_name TEXT, cast_order INTEGER,
FOREIGN KEY(movie_id) REFERENCES movies(movie_id))''',
'''CREATE TABLE crew (
crew_row_id INTEGER PRIMARY KEY, movie_id INTEGER, person_id INTEGER,
person_name TEXT, job TEXT, department TEXT,
FOREIGN KEY(movie_id) REFERENCES movies(movie_id))''',
'''CREATE TABLE movie_keywords (
movie_id INTEGER, keyword_id INTEGER, keyword_name TEXT,
PRIMARY KEY(movie_id,keyword_id),
FOREIGN KEY(movie_id) REFERENCES movies(movie_id))'''
]
for statement in ddl:
    conn.execute(statement)
for table, df in data.items():
    df.to_sql(table, conn, if_exists="append", index=False)
conn.commit()
print("Database created:", DB_PATH)

## 4. Verify row counts and foreign-key integrity

In [ ]:
for table in tables:
    count = pd.read_sql_query(f"SELECT COUNT(*) AS n FROM {table}", conn).iloc[0,0]
    print(table, count)
print("Foreign-key check:", conn.execute("PRAGMA foreign_key_check").fetchall())

## Q1 – Action movies released after 2015

In [ ]:
query = """\nSELECT m.title, m.release_date
FROM movies m JOIN movie_genres mg ON m.movie_id=mg.movie_id
JOIN genres g ON mg.genre_id=g.genre_id
WHERE g.genre_name='Action' AND m.release_date > '2015-12-31'
ORDER BY m.release_date;\n"""\nresult = pd.read_sql_query(query, conn)\ndisplay(result)\nprint("Rows returned:", len(result))

## Q2 – Top 10 highest-grossing movies with genres

In [ ]:
query = """\nSELECT m.title, m.revenue, GROUP_CONCAT(g.genre_name, ', ') AS genres
FROM movies m JOIN movie_genres mg ON m.movie_id=mg.movie_id
JOIN genres g ON mg.genre_id=g.genre_id
WHERE m.revenue>0
GROUP BY m.movie_id,m.title,m.revenue
ORDER BY m.revenue DESC LIMIT 10;\n"""\nresult = pd.read_sql_query(query, conn)\ndisplay(result)\nprint("Rows returned:", len(result))

## Q3 – Average budget and revenue by genre

In [ ]:
query = """\nSELECT g.genre_name,
COUNT(DISTINCT m.movie_id) AS total_movies,
COUNT(DISTINCT CASE WHEN m.budget>0 THEN m.movie_id END) AS movies_with_budget,
COUNT(DISTINCT CASE WHEN m.revenue>0 THEN m.movie_id END) AS movies_with_revenue,
ROUND(AVG(NULLIF(m.budget,0)),2) AS average_budget,
ROUND(AVG(NULLIF(m.revenue,0)),2) AS average_revenue
FROM movies m JOIN movie_genres mg ON m.movie_id=mg.movie_id
JOIN genres g ON mg.genre_id=g.genre_id
GROUP BY g.genre_id,g.genre_name ORDER BY average_revenue DESC;\n"""\nresult = pd.read_sql_query(query, conn)\ndisplay(result)\nprint("Rows returned:", len(result))

## Q4 – Top 10 most prolific actors

In [ ]:
query = """\nSELECT person_id,actor_name,COUNT(DISTINCT movie_id) AS movie_count
FROM cast GROUP BY person_id,actor_name
ORDER BY movie_count DESC,actor_name LIMIT 10;\n"""\nresult = pd.read_sql_query(query, conn)\ndisplay(result)\nprint("Rows returned:", len(result))

## Q5 – Directors with more than three movies

In [ ]:
query = """\nSELECT person_id,person_name AS director_name,
COUNT(DISTINCT movie_id) AS movie_count
FROM crew WHERE job='Director'
GROUP BY person_id,person_name
HAVING COUNT(DISTINCT movie_id)>3
ORDER BY movie_count DESC,director_name;\n"""\nresult = pd.read_sql_query(query, conn)\ndisplay(result)\nprint("Rows returned:", len(result))

## Q6 – Top 10 most common keywords

In [ ]:
query = """\nSELECT keyword_id,keyword_name,COUNT(DISTINCT movie_id) AS movie_count
FROM movie_keywords GROUP BY keyword_id,keyword_name
ORDER BY movie_count DESC,keyword_name LIMIT 10;\n"""\nresult = pd.read_sql_query(query, conn)\ndisplay(result)\nprint("Rows returned:", len(result))

## Q7 – Above-average budget and below-average revenue

In [ ]:
query = """\nWITH averages AS (
SELECT AVG(NULLIF(budget,0)) avg_budget, AVG(NULLIF(revenue,0)) avg_revenue FROM movies
)
SELECT m.movie_id,m.title,m.budget,m.revenue,
ROUND(a.avg_budget,2) overall_avg_budget,ROUND(a.avg_revenue,2) overall_avg_revenue
FROM movies m CROSS JOIN averages a
WHERE m.budget>a.avg_budget AND m.revenue>0 AND m.revenue<a.avg_revenue
ORDER BY m.budget DESC;\n"""\nresult = pd.read_sql_query(query, conn)\ndisplay(result)\nprint("Rows returned:", len(result))

## Q8 – Actors in movies directed by Christopher Nolan

In [ ]:
query = """\nSELECT DISTINCT c.person_id,c.actor_name,m.title AS movie_title
FROM movies m JOIN cast c ON m.movie_id=c.movie_id
JOIN crew cr ON m.movie_id=cr.movie_id
WHERE cr.person_name='Christopher Nolan' AND cr.job='Director'
ORDER BY m.title,c.actor_name;\n"""\nresult = pd.read_sql_query(query, conn)\ndisplay(result)\nprint("Rows returned:", len(result))

## Q9 – Highest-rated genre with at least 20 movies

In [ ]:
query = """\nSELECT g.genre_name,COUNT(DISTINCT m.movie_id) AS movie_count,
ROUND(AVG(m.vote_average),3) AS average_rating
FROM movies m JOIN movie_genres mg ON m.movie_id=mg.movie_id
JOIN genres g ON mg.genre_id=g.genre_id
GROUP BY g.genre_id,g.genre_name
HAVING COUNT(DISTINCT m.movie_id)>=20
ORDER BY average_rating DESC LIMIT 1;\n"""\nresult = pd.read_sql_query(query, conn)\ndisplay(result)\nprint("Rows returned:", len(result))

## Q10 – Top 10 movies with most keywords

In [ ]:
query = """\nSELECT m.movie_id,m.title,COUNT(DISTINCT mk.keyword_id) AS keyword_count
FROM movies m JOIN movie_keywords mk ON m.movie_id=mk.movie_id
GROUP BY m.movie_id,m.title
ORDER BY keyword_count DESC,m.title LIMIT 10;\n"""\nresult = pd.read_sql_query(query, conn)\ndisplay(result)\nprint("Rows returned:", len(result))

## SQL analysis summary \n
The 10 queries cover genre releases, revenue, genre financial averages, actors, directors, keywords, financial comparisons, director collaborations and ratings. Interpret financial comparisons carefully: budget and revenue alone do not represent full profitability.